# World of Shadow Work — high-quality vessel splats (`vessel.wswv`)

Generates **128** high-quality image-to-3D Gaussian splats for the runtime's morphing "vessel", with **TRELLIS** (Microsoft image-to-3D structured-latent — best 3DGS quality). One splat keyframe per input image; the runtime melts between them live.

**Runtime: A100 80GB.** TRELLIS needs ~16–24 GB, so 80 GB runs it at full sampling steps and batches comfortably.

**Paths + corpus mirror `wosw_dream_colab.ipynb` (the one that worked):** clone the `world-of-shadow-work` branch, `%cd` into `reagency/factory`, **mount Drive and read the corpus from `DRIVE_CORPUS`** (the AIC source URLs 403 on direct fetch — Drive reuse avoids that), work under `work/` (`work/vessel_inputs/` inputs → `work/vessels/*.ply`), pack to `../assets/vessel.wswv`, and **download to the Mac** (never git-push — factory rule).

**Pairs with the runtime:** `DENS=4` in `viz/VesselSplats.cpp` (already set) for these dense `G=30000` clouds. On the Mac: `cp ~/Downloads/vessel.wswv reagency/assets/vessel.wswv` — the runtime auto-loads it.

## 0 · Config (one source of truth)

In [ ]:
REPO     = '9LiveZZZ-Git/MAT201B_Projects'
BRANCH   = 'world-of-shadow-work'   # matches wosw_dream_colab (the path that worked)

# Corpus from Google Drive (AIC source URLs 403 on direct fetch — same reuse as wosw_dream_colab).
USE_DRIVE_CORPUS = True
DRIVE_CORPUS     = '/content/drive/MyDrive/wosw/corpus.zip'   # a .zip OR a folder on your Drive

N_IMAGES = 128      # splat keyframes to generate (one per input image)
SELECT   = 'spread' # which corpus images: 'spread' (evenly across corpus) | 'first' (incl. the dream sources)
STEPS    = 50       # TRELLIS sampling steps (sparse-structure + SLAT); 50 = high quality
G        = 30000    # gaussians/keyframe — pair with runtime DENS=4 (VesselSplats.cpp)
MAX_MB   = 80       # vessel.wswv size cap (128 x 30000 x 16 B ≈ 61 MB; keep above that so G holds)
print('config:', N_IMAGES, 'keyframes | G', G, '| STEPS', STEPS, '| corpus:', DRIVE_CORPUS)

## 1 · Confirm the A100 80GB
Runtime → Change runtime type → **A100 GPU**. Expect `A100-SXM4-80GB`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0))

## 2 · Clone the branch + cd into the factory + mount Drive  (mirrors wosw_dream_colab cell 6)

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/MAT201B_Projects'):
    !git clone -b {BRANCH} https://github.com/{REPO}.git
%cd /content/MAT201B_Projects/reagency/factory
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('work/vessel_inputs', exist_ok=True)
os.makedirs('work/vessels', exist_ok=True)
print('cwd:', os.getcwd())

## 3 · Input images → `work/vessel_inputs/`  (from the Drive corpus — NO URL fetch)
Reuses `work/vessel_inputs/*.jpg` if already populated; else restores the corpus from `DRIVE_CORPUS` (zip → unzip, folder → symlink, exactly like wosw_dream_colab cell 10) and copies `N_IMAGES` of `corpus/images/**/*.jpg` in (`sorted` = the same node order as `stage_a_embed.py`).

In [ ]:
import os, glob, shutil
INP = 'work/vessel_inputs'
existing = sorted(glob.glob(f'{INP}/*.jpg') + glob.glob(f'{INP}/*.png'))
if len(existing) >= N_IMAGES:
    print(f'reusing {len(existing)} existing inputs in {INP}/')
else:
    # corpus from Drive (zip -> unzip, folder -> symlink) — avoids the AIC 403s
    if USE_DRIVE_CORPUS and not os.path.isdir('corpus/images'):
        !rm -rf corpus _cz
        if DRIVE_CORPUS.endswith('.zip'):
            !unzip -q -o "{DRIVE_CORPUS}" -d _cz
            shutil.move('_cz/corpus' if os.path.isdir('_cz/corpus') else '_cz', 'corpus')
        else:
            os.symlink(DRIVE_CORPUS, 'corpus')
    imgs = sorted(glob.glob('corpus/images/**/*.jpg', recursive=True))   # same glob/order as stage_a_embed.py:59
    assert imgs, 'empty corpus — set USE_DRIVE_CORPUS / DRIVE_CORPUS (a .zip or folder on your Drive)'
    print('corpus images on Drive:', len(imgs))
    if SELECT == 'first': picks = imgs[:N_IMAGES]
    else:                 picks = imgs[::max(1, len(imgs)//N_IMAGES)][:N_IMAGES]   # 'spread'
    for p in picks:
        shutil.copy(p, f'{INP}/{os.path.basename(p)}')
    print('inputs ready:', len(glob.glob(f'{INP}/*')))

## 4 · Install TRELLIS  (~5–10 min first run; comfortable on the 80GB A100)
Repo: https://github.com/microsoft/TRELLIS . If a backend flag fails to build, drop it (e.g. `--flash-attn`) — the run cell uses `xformers`. (The `_Ink` / Pillow clash is fixed in the next cell.)

In [ ]:
import os
if not os.path.isdir('/content/TRELLIS'):
    !git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
!. ./setup.sh --basic --xformers --spconv --mipgaussian --nvdiffrast --diffoctreerast --kaolin
!pip -q install rembg onnxruntime imageio imageio-ffmpeg plyfile
%cd /content/MAT201B_Projects/reagency/factory

## 5 · Run TRELLIS → one 3DGS `.ply` per image  →  `work/vessels/`
**Self-heals the `torchvision`/Pillow `_Ink` ImportError** (a half-upgraded Pillow): first run force-reinstalls a **single consistent Pillow ≥12.1.0** (also what `rembg` requires) and **restarts the runtime once** — then just **re-run this cell** and it generates. Re-runnable (existing `.ply`s skipped).

In [ ]:
# === Run TRELLIS image->3DGS  (self-heals the torchvision/Pillow `_Ink` ImportError) ===
import os, sys, glob

# The `_Ink` error is a HALF-UPGRADED Pillow (mixed version files). Force-reinstall ONE consistent
# Pillow that ALSO satisfies rembg (>=12.1.0,<13), then restart once so the kernel reloads it clean.
_OK = '/content/.wsw_pillow_ok2'
if not os.path.exists(_OK):
    os.system('pip -q install --force-reinstall "pillow>=12.1.0,<13"')
    open(_OK, 'w').close()
    print('Pillow reinstalled (>=12.1.0, rembg-compatible) — RESTARTING. Re-run THIS cell after restart.')
    os.kill(os.getpid(), 9)   # Colab restarts the kernel here; just run the cell again

# robust even if you only re-run this cell after the restart (cwd resets; STEPS may be undefined)
os.chdir('/content/MAT201B_Projects/reagency/factory')
try: STEPS
except NameError: STEPS = 50

os.environ['ATTN_BACKEND'] = 'xformers'   # 'flash-attn' if you built it
os.environ['SPCONV_ALGO']  = 'native'
from PIL import Image
sys.path.insert(0, '/content/TRELLIS')
from trellis.pipelines import TrellisImageTo3DPipeline

pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large'); pipe.cuda()
imgs = sorted(glob.glob('work/vessel_inputs/*.jpg') + glob.glob('work/vessel_inputs/*.png'))
print(f'{len(imgs)} images -> splats (STEPS={STEPS})')
for i, ip in enumerate(imgs):
    name = os.path.splitext(os.path.basename(ip))[0]; out = f'work/vessels/{name}.ply'
    if os.path.exists(out) and os.path.getsize(out) > 0:
        continue
    try:
        outputs = pipe.run(Image.open(ip).convert('RGB'), seed=1,
                           sparse_structure_sampler_params={'steps': STEPS, 'cfg_strength': 7.5},
                           slat_sampler_params={'steps': STEPS, 'cfg_strength': 3.0})
        outputs['gaussian'][0].save_ply(out)
        print(f'[{i+1}/{len(imgs)}] {name}.ply  {os.path.getsize(out)/1e6:.1f} MB')
    except Exception as e:
        print(f'[{i+1}/{len(imgs)}] SKIP {name}: {e}')
print('done -> work/vessels/', len(glob.glob('work/vessels/*.ply')), 'plys')

## 6 · Pack → `../assets/vessel.wswv`  (existing Stage-D packer)
Prunes each keyframe to a fixed `G`, normalizes to one global AABB, Morton-orders → a real morph (not swimming). `--max-mb {MAX_MB}` keeps `G=30000` from being auto-reduced.

In [ ]:
!python3 stage_d_vessel.py --plys work/vessels --G {G} --max-mb {MAX_MB}
import os
print('vessel.wswv:', round(os.path.getsize('../assets/vessel.wswv')/1e6, 2), 'MB')

## 7 · Download to the Mac  (NEVER git-push from Colab)
Then: `cp ~/Downloads/vessel.wswv MAT201B_Projects/reagency/assets/vessel.wswv` — the runtime tries `assets/vessel.wswv` first, so it auto-loads next launch. (`DENS=4` is already set in `VesselSplats.cpp` for these dense clouds.)

In [ ]:
import os, glob
from google.colab import files
cands = ['../assets/vessel.wswv', '/content/MAT201B_Projects/reagency/assets/vessel.wswv']
OUT = next((p for p in cands if os.path.isfile(p)), None)
if OUT is None:
    hits = glob.glob('/content/**/vessel.wswv', recursive=True); OUT = hits[0] if hits else None
assert OUT, "vessel.wswv not found — did Stage D (cell 6) run? try: !find / -name vessel.wswv 2>/dev/null"
print('downloading %s (%.1f MB)...' % (OUT, os.path.getsize(OUT)/1e6))
files.download(OUT)
# --- or stage on Drive instead of a browser download: ---
# import shutil; os.makedirs('/content/drive/MyDrive/wosw', exist_ok=True)
# shutil.copy(OUT, '/content/drive/MyDrive/wosw/vessel.wswv'); print('staged on Drive')

## Fallback — LGM (if TRELLIS install fails)
Feed-forward, seconds/image on an A100; emits 3DGS `.ply` that `stage_d_vessel.py` reads the same way. Run instead of cells 4–5, then continue at cell 6.
```bash
%cd /content
!pip -q install -U xformers plyfile
!git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization && pip -q install ./diff-gaussian-rasterization
!pip -q install git+https://github.com/NVlabs/nvdiffrast
!git clone https://github.com/3DTopia/LGM && cd LGM && pip -q install -r requirements.txt
# download pretrained/model_fp16.safetensors per the LGM README, then:
!cd LGM && python infer.py big --resume pretrained/model_fp16.safetensors \
    --workspace /content/MAT201B_Projects/reagency/factory/work/vessels \
    --test_path /content/MAT201B_Projects/reagency/factory/work/vessel_inputs
```
(Full LGM runbook in `factory/VESSEL.md`.)